In [7]:
from pathlib import Path

import pandas as pd
import numpy as np
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from transformers import AutoImageProcessor, AutoModelForImageClassification
from tqdm.auto import tqdm
from sklearn.cluster import KMeans

#sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

DATA_DIR = Path("geo_dataset")  # change this
TRAIN_DIR = DATA_DIR / "train"
HOLDOUT_DIR = DATA_DIR / "holdout_public"
LABELS_PATH = DATA_DIR / "train_labels.csv"

In [8]:
OUTPUT_DIR = Path("outputs/geographic_cells")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

best_model_path = OUTPUT_DIR / "best_model.pt"
checkpoint_path = OUTPUT_DIR / "training_checkpoint.pt"
history_path = OUTPUT_DIR / "history.csv"

In [9]:
df = pd.read_csv(LABELS_PATH)

print(df.shape)
display(df.head())

(11758, 5)


,filename,country,iso,lat,lng
0,1fcb4a43864244259b7d8f4a00f1e475.jpg,Turkey,TR,40.112290,38.304629
1,742f45b0211c44ffb19ad84931ea519c.jpg,France,FR,48.094103,-1.994316
2,152a13ef249d4efa95c51ed93f026284.jpg,Turkey,TR,41.324741,27.961821
3,81ce4a88bff14fef8420bca42019b12b.jpg,France,FR,47.585855,-2.971004
4,6fbcfe523e1349759e6060d632d52e54.jpg,United_Kingdom,GB,55.698094,-4.305315


Validation Split

In [10]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["country"],
)

print("Training images:", len(train_df))
print("Validation images:", len(val_df))

Training images: 9406
Validation images: 2352


Creating a 64 Cell Grid over Europe

In [11]:
NUMBER_OF_CELLS = 64

kmeans = KMeans(
    n_clusters=NUMBER_OF_CELLS,
    random_state=42,
    n_init=10,
)

kmeans.fit(
    train_df[["lat", "lng"]]
)

train_df = train_df.copy()
val_df = val_df.copy()

train_df["cell_index"] = kmeans.labels_

val_df["cell_index"] = kmeans.predict(
    val_df[["lat", "lng"]]
)

cell_centres = (
    kmeans.cluster_centers_
    .astype(np.float32)
)

print("Cell centres:", cell_centres.shape)

Cell centres: (64, 2)


In [12]:
cell_counts = (
    train_df["cell_index"]
    .value_counts()
    .sort_index()
)

display(cell_counts)

print("Smallest cell:", cell_counts.min())
print("Largest cell:", cell_counts.max())

cell_index
0      70
1     151
2     157
3     184
4     205
     ... 
59    216
60    119
61    100
62    155
63     91
Name: count, Length: 64, dtype: int64

Smallest cell: 63
Largest cell: 289


Model Verification

In [14]:
MODEL_NAME = (
    "apple/mobilevitv2-1.0-imagenet1k-256"
)

processor = AutoImageProcessor.from_pretrained(
    MODEL_NAME
)

model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2 + NUMBER_OF_CELLS,
    ignore_mismatched_sizes=True,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Device:", device)

[transformers] You passed `num_labels=66` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 269/269 [00:00<00:00, 45040.63it/s]
[transformers] MobileViTV2ForImageClassification LOAD REPORT from: apple/mobilevitv2-1.0-imagenet1k-256
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([66])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([66, 512])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Device: cuda


In [15]:
total_params = sum(p.numel() for p in model.parameters())

print(f"Parameters: {total_params:,}")
assert total_params <= 5_000_000

Parameters: 4,422,699


Normalizing grid cell centres

In [16]:
normalized_cell_centres = (
    cell_centres.copy()
)

normalized_cell_centres[:, 0] /= 90
normalized_cell_centres[:, 1] /= 180

In [17]:
cell_centres_tensor = torch.tensor(
    normalized_cell_centres,
    dtype=torch.float32,
    device=device,
)

In [18]:
print(cell_centres_tensor.shape)

torch.Size([64, 2])


Image Processor and Dataset

In [19]:
class GeolocationDataset(Dataset):
    def __init__(
        self,
        dataframe,
        image_dir,
        processor,
        transform=None,
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.processor = processor
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = self.image_dir / row["filename"]
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        pixel_values = self.processor(
            images=image,
            return_tensors="pt",
        )["pixel_values"].squeeze(0)

        coordinates = torch.tensor(
            [
                row["lat"] / 90,
                row["lng"] / 180,
            ],
            dtype=torch.float32,
        )

        cell_index = torch.tensor(
            row["cell_index"],
            dtype=torch.long,
        )

        return pixel_values, coordinates, cell_index

In [20]:
train_dataset = GeolocationDataset(
    train_df,
    TRAIN_DIR,
    processor,
    transform=None,
)

val_dataset = GeolocationDataset(
    val_df,
    TRAIN_DIR,
    processor,
    transform=None,
)

In [21]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

#Loss and Optimizer

In [24]:
coordinate_loss_function = nn.MSELoss()
cell_loss_function = nn.CrossEntropyLoss()

CELL_LOSS_WEIGHT = 0.01
COORDINATE_WEIGHT = 0.25

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
)

In [23]:
images, coordinates, cell_labels = next(
    iter(train_loader)
)

images = images.to(device)
coordinates = coordinates.to(device)
cell_labels = cell_labels.to(device)

print("Images:", images.shape)
print("Coordinates:", coordinates.shape)
print("Cell labels:", cell_labels.shape)

Images: torch.Size([32, 3, 256, 256])
Coordinates: torch.Size([32, 2])
Cell labels: torch.Size([32])


In [25]:
def haversine_km(lat1, lng1, lat2, lng2):
    radius = 6371.0088

    lat1 = np.radians(lat1)
    lng1 = np.radians(lng1)
    lat2 = np.radians(lat2)
    lng2 = np.radians(lng2)

    difference = (
        np.sin((lat2 - lat1) / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin((lng2 - lng1) / 2) ** 2
    )

    return (
        2
        * radius
        * np.arcsin(
            np.sqrt(np.clip(difference, 0, 1))
        )
    )

Full Train + Validation Loop (Current best-> Epochs: 20, median: 709) (Reload Optimizer before continuing training)

In [28]:
start_epoch = 0
END_EPOCH = 40

history = []

best_median = float("inf")
best_epoch = 0

epochs_without_improvement = 0
patience = 4

for epoch in range(start_epoch, END_EPOCH):

    # --------------------
    # Training
    # --------------------
    model.train()
    total_training_loss = 0

    training_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{END_EPOCH} - Training",
    )

    for images, coordinates, cell_labels in training_bar:
        images = images.to(device)
        coordinates = coordinates.to(device)
        cell_labels = cell_labels.to(device)

        optimizer.zero_grad()

        outputs = model(
            pixel_values=images
        ).logits

        #Split the outputs
        direct_coordinates = torch.tanh(
            outputs[:, :2]
        )

        cell_logits = outputs[:, 2:]

        cell_probabilities = torch.softmax(
            cell_logits,
            dim=1,
        )

        cell_coordinates = (
            cell_probabilities
            @ cell_centres_tensor
        )

        final_coordinates = (
            COORDINATE_WEIGHT
            * direct_coordinates
            + (1 - COORDINATE_WEIGHT)
            * cell_coordinates
        )

        coordinate_loss = coordinate_loss_function(
            final_coordinates,
            coordinates,
        )

        cell_loss = cell_loss_function(
            cell_logits,
            cell_labels,
        )

        loss = (
            coordinate_loss
            + CELL_LOSS_WEIGHT * cell_loss
        )

        loss.backward()
        optimizer.step()

        total_training_loss += loss.item()

        training_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    average_training_loss = (
        total_training_loss / len(train_loader)
    )

    # --------------------
    # Validation
    # --------------------
    model.eval()

    total_validation_loss = 0
    all_predictions = []
    all_coordinates = []

    validation_bar = tqdm(
        val_loader,
        desc=f"Epoch {epoch + 1}/{END_EPOCH} - Validation",
    )

    correct_cell_predictions = 0
    number_of_validation_images = 0

    with torch.no_grad():
        for images, coordinates, cell_labels in validation_bar:
            images = images.to(device)
            coordinates = coordinates.to(device)
            cell_labels = cell_labels.to(device)

            outputs = model(
                pixel_values=images
            ).logits

            # Direct coordinate prediction
            direct_coordinates = torch.tanh(
                outputs[:, :2]
            )

            # Cell prediction
            cell_logits = outputs[:, 2:]

            cell_probabilities = torch.softmax(
                cell_logits,
                dim=1,
            )

            # Probability-weighted cell coordinate
            cell_coordinates = (
                cell_probabilities
                @ cell_centres_tensor
            )

            # Final blended coordinate
            final_coordinates = (
                COORDINATE_WEIGHT
                * direct_coordinates
                + (1 - COORDINATE_WEIGHT)
                * cell_coordinates
            )

            # Evaluate the same blended coordinate used during training
            coordinate_loss = coordinate_loss_function(
                final_coordinates,
                coordinates,
            )

            cell_loss = cell_loss_function(
                cell_logits,
                cell_labels,
            )

            loss = (
                coordinate_loss
                + CELL_LOSS_WEIGHT * cell_loss
            )

            total_validation_loss += loss.item()

            # Store the final blended prediction, not the direct prediction
            all_predictions.append(
                final_coordinates.cpu().numpy()
            )

            all_coordinates.append(
                coordinates.cpu().numpy()
            )

            predicted_cells = cell_logits.argmax(
                dim=1
            )

            correct_cell_predictions += (
                predicted_cells == cell_labels
            ).sum().item()

            number_of_validation_images += (
                cell_labels.size(0)
            )     

    average_validation_loss = (
        total_validation_loss / len(val_loader)
    )

    cell_accuracy = (
        correct_cell_predictions
        / number_of_validation_images
    )


    # --------------------
    # Geographic metrics
    # --------------------
    all_predictions = np.concatenate(
        all_predictions
    )

    all_coordinates = np.concatenate(
        all_coordinates
    )

    predictions_degrees = all_predictions.copy()
    coordinates_degrees = all_coordinates.copy()

    predictions_degrees[:, 0] *= 90
    predictions_degrees[:, 1] *= 180

    coordinates_degrees[:, 0] *= 90
    coordinates_degrees[:, 1] *= 180

    distances = haversine_km(
        coordinates_degrees[:, 0],
        coordinates_degrees[:, 1],
        predictions_degrees[:, 0],
        predictions_degrees[:, 1],
    )

    mean_distance = np.mean(distances)
    median_distance = np.median(distances)
    within_200 = np.mean(distances < 200)
    within_750 = np.mean(distances < 750)

    history.append({
        "epoch": epoch + 1,
        "training_loss": average_training_loss,
        "validation_loss": average_validation_loss,
        "mean_km": mean_distance,
        "median_km": median_distance,
        "within_200": within_200,
        "within_750": within_750,
        "cell_accuracy": cell_accuracy,
    })

    print(f"\nEpoch {epoch + 1} results")
    print(f"Training loss: {average_training_loss:.4f}")
    print(f"Validation loss: {average_validation_loss:.4f}")
    print(f"Mean distance: {mean_distance:.1f} km")
    print(f"Median distance: {median_distance:.1f} km")
    print(f"Within 200 km: {within_200:.2%}")
    print(f"Within 750 km: {within_750:.2%}")
    print(f"Cell accuracy: {cell_accuracy:.2%}")

    # --------------------
    # Best model
    # --------------------
    if median_distance < best_median:
        best_median = median_distance
        best_epoch = epoch + 1

        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            best_model_path,
        )

        print("Saved new best model.")

    else:
        epochs_without_improvement += 1

        print(
            "Epochs without improvement:",
            epochs_without_improvement,
        )

    # Save latest resumable checkpoint
    torch.save(
        {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_median": best_median,
            "best_epoch": best_epoch,
            "history": history,
            "patience": patience,
            "epochs_without_improvement": (
                epochs_without_improvement
            ),
            "model_name": MODEL_NAME,
            "num_labels": 2 + NUMBER_OF_CELLS,
            "parameter_count": total_params,
            "number_of_cells": NUMBER_OF_CELLS,
            "cell_centres": (
                cell_centres_tensor
                .detach()
                .cpu()
            ),
            "coordinate_weight": COORDINATE_WEIGHT,
            "cell_loss_weight": CELL_LOSS_WEIGHT,
        },
        checkpoint_path,
    )

    # Preserve history after every epoch
    pd.DataFrame(history).to_csv(
        history_path,
        index=False,
    )

    if epochs_without_improvement >= patience:
        print("Early stopping.")
        break

Epoch 1/40 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.19it/s]



Epoch 1 results
Training loss: 0.0525
Validation loss: 0.0438
Mean distance: 1071.2 km
Median distance: 974.0 km
Within 200 km: 2.25%
Within 750 km: 33.50%
Cell accuracy: 11.95%
Saved new best model.


Epoch 2/40 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.53it/s]



Epoch 2 results
Training loss: 0.0386
Validation loss: 0.0362
Mean distance: 843.3 km
Median distance: 703.7 km
Within 200 km: 7.23%
Within 750 km: 53.23%
Cell accuracy: 19.30%
Saved new best model.


Epoch 3/40 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.67it/s]


...


Epoch 26 results
Training loss: 0.0008
Validation loss: 0.0412
Mean distance: 588.9 km
Median distance: 361.9 km
Within 200 km: 32.57%
Within 750 km: 71.98%
Cell accuracy: 38.31%
Epochs without improvement: 2


Epoch 27/40 - Validation: 100%|██████████| 74/74 [00:12<00:00,  6.07it/s]



Epoch 27 results
Training loss: 0.0008
Validation loss: 0.0416
Mean distance: 595.9 km
Median distance: 364.6 km
Within 200 km: 32.91%
Within 750 km: 71.26%
Cell accuracy: 38.65%
Epochs without improvement: 3


Epoch 28/40 - Validation: 100%|██████████| 74/74 [00:12<00:00,  6.13it/s]



Epoch 28 results
Training loss: 0.0008
Validation loss: 0.0419
Mean distance: 591.1 km
Median distance: 364.5 km
Within 200 km: 32.06%
Within 750 km: 71.73%
Cell accuracy: 38.86%
Epochs without improvement: 4
Early stopping.


Diagnostic

In [34]:
best_weights = torch.load(
    best_model_path,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(best_weights)
model.eval()

MobileViTV2ForImageClassification(
  (mobilevitv2): MobileViTV2Model(
    (conv_stem): MobileViTV2ConvLayer(
      (convolution): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (normalization): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (activation): SiLU()
    )
    (encoder): MobileViTV2Encoder(
      (layer): ModuleList(
        (0): MobileViTV2MobileNetLayer(
          (layer): ModuleList(
            (0): MobileViTV2InvertedResidual(
              (expand_1x1): MobileViTV2ConvLayer(
                (convolution): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
                (normalization): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
                (activation): SiLU()
              )
              (conv_3x3): MobileViTV2ConvLayer(
                (convolution): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), gr

In [35]:
all_direct_coordinates = []
all_cell_probabilities = []
all_true_coordinates = []

model.eval()

with torch.no_grad():
    for images, coordinates, cell_labels in tqdm(
        val_loader,
        desc="Testing blend weights",
    ):
        images = images.to(device)

        outputs = model(
            pixel_values=images
        ).logits

        direct_coordinates = torch.tanh(
            outputs[:, :2]
        )

        cell_probabilities = torch.softmax(
            outputs[:, 2:],
            dim=1,
        )

        all_direct_coordinates.append(
            direct_coordinates.cpu().numpy()
        )

        all_cell_probabilities.append(
            cell_probabilities.cpu().numpy()
        )

        all_true_coordinates.append(
            coordinates.numpy()
        )

Testing blend weights: 100%|██████████| 74/74 [00:11<00:00,  6.20it/s]


In [36]:
all_direct_coordinates = np.concatenate(
    all_direct_coordinates
)

all_cell_probabilities = np.concatenate(
    all_cell_probabilities
)

all_true_coordinates = np.concatenate(
    all_true_coordinates
)

In [37]:
soft_cell_coordinates = (
    all_cell_probabilities
    @ normalized_cell_centres
)

In [38]:
blend_results = []

for coordinate_weight in [
    0.0,
    0.10,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
    0.75,
    1.0,
]:
    final_coordinates = (
        coordinate_weight
        * all_direct_coordinates
        + (1 - coordinate_weight)
        * soft_cell_coordinates
    )

    predictions_degrees = final_coordinates.copy()
    true_degrees = all_true_coordinates.copy()

    predictions_degrees[:, 0] *= 90
    predictions_degrees[:, 1] *= 180

    true_degrees[:, 0] *= 90
    true_degrees[:, 1] *= 180

    distances = haversine_km(
        true_degrees[:, 0],
        true_degrees[:, 1],
        predictions_degrees[:, 0],
        predictions_degrees[:, 1],
    )

    blend_results.append({
        "coordinate_weight": coordinate_weight,
        "mean_km": np.mean(distances),
        "median_km": np.median(distances),
        "within_200": np.mean(distances < 200),
        "within_750": np.mean(distances < 750),
    })

In [39]:
display(
    pd.DataFrame(blend_results)
)

,coordinate_weight,mean_km,median_km,within_200,within_750
0,0.00,585.088196,333.777435,0.374150,0.721088
1,0.10,581.749634,345.829041,0.367347,0.722364
2,0.20,585.392029,351.664734,0.343963,0.721939
3,0.25,589.371704,357.673859,0.325255,0.722364
4,0.30,594.531372,373.234070,0.307823,0.719813
5,0.40,607.885498,393.226562,0.267432,0.715136
6,0.50,624.670166,417.985840,0.232993,0.706633
7,0.75,677.927917,498.930481,0.164966,0.672619
8,1.00,743.728821,580.082092,0.109694,0.619898
